In [1]:
import mlxtend
from mlxtend.frequent_patterns import apriori, association_rules

print("mlxtend working!")



mlxtend working!


In [4]:
# ASSOCIATION RULE MINING MODULE

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. LOAD DATA
file = r"C:\Users\HP\Data Science\eda_final_project\Sales.csv"
sales = pd.read_csv(file)

print("Sales Data Loaded Successfully!")
print(sales.head())

# 2. CONVERT ROWS TO TRANSACTION BASKET

def make_basket(row):
    basket = []

    # Brand
    basket.append(f"brand_{row['brand']}")

    # RAM
    basket.append(f"ram_{int(row['ram_gb'])}GB")

    # Storage
    basket.append(f"storage_{int(row['storage_gb'])}GB")

    # PRICE CATEGORY
    p = row["base_price"]
    if p < 10000:
        pr = "Low"
    elif p < 20000:
        pr = "Mid"
    elif p < 30000:
        pr = "High"
    else:
        pr = "Premium"

    basket.append(f"price_{pr}")

# FIXED: DISCOUNT AMOUNT (correct columns)

    discount_amount = row["base_price"] - row["final_price"]

    if discount_amount > 5000:
        basket.append("High_Discount")
    elif discount_amount > 2000:
        basket.append("Medium_Discount")
    else:
        basket.append("Low_Discount")

    return basket


transactions = sales.apply(make_basket, axis=1).tolist()

print("\nSample Transaction:", transactions[0])

# 3. ONE-HOT ENCODING

te = TransactionEncoder()
encoded = te.fit(transactions).transform(transactions)
df = pd.DataFrame(encoded, columns=te.columns_)

# 4. APRIORI

frequent = apriori(df, min_support=0.05, use_colnames=True)

# 5. ASSOCIATION RULES

rules = association_rules(frequent, metric="lift", min_threshold=1.0)
rules = rules.sort_values("confidence", ascending=False)

print("\nRules Generated:", len(rules))

# 6. BRAND + STORAGE ⇒ HIGH DISCOUNT

print("\n=== Brand + Storage ⇒ High Discount ===")

brand_storage_hd = rules[
    rules["consequents"].astype(str).str.contains("High_Discount")
]

print(brand_storage_hd[[
    "antecedents", "consequents", "support", "confidence", "lift"
]].head(10))

# 7. RAM ⇒ STORAGE

print("\n=== RAM ⇒ Storage ===")

ram_storage = rules[
    rules["antecedents"].astype(str).str.contains("ram_") &
    rules["consequents"].astype(str).str.contains("storage_")
]

print(ram_storage[[
    "antecedents", "consequents", "support", "confidence", "lift"
]].head(10))

# 8. BRAND ⇒ PRICE

print("\n=== Brand ⇒ Price Category ===")

brand_price = rules[
    rules["antecedents"].astype(str).str.contains("brand_") &
    rules["consequents"].astype(str).str.contains("price_")
]

print(brand_price[[
    "antecedents", "consequents", "support", "confidence", "lift"
]].head(10))

# 9. TOP 20 RULES

print("\n=== Top 20 Rules ===")

print(rules.head(20)[[
    "antecedents", "consequents", "support", "confidence", "lift"
]])


Sales Data Loaded Successfully!
    sale_id customer_id model_id   shop_id   sale_date  quantity  discount  \
0  SALE4000    CUST2083  MOD2111  SHOP3068  2024-11-26         1      1.05   
1  SALE4001    CUST2384  MOD2021  SHOP3063  2021-07-13         1     10.48   
2  SALE4002    CUST6189  MOD2008  SHOP3004  2025-05-04         2     13.55   
3  SALE4003    CUST4992  MOD2105  SHOP3041  2023-02-28         1      7.57   
4  SALE4004    CUST4707  MOD2109  SHOP3024  2020-04-12         1     18.79   

   final_price payment_mode     status     brand model_name  ram_gb  \
0   22316.1935  Credit Card  Delivered    realme    Series6       4   
1   26218.6176         Cash    Pending   Samsung   Series16       4   
2   36469.7970         Cash  Cancelled     Apple   Series19       6   
3   37230.8040  Credit Card    Pending     Apple    Series2      12   
4   35201.2866          UPI  Delivered  Motorola    Series8       6   

   storage_gb  base_price     offer_id  
0          64     22553.0  OFF_